# SmartTraffic — Camera-to-Forecast Data Preparation and Model Training

This notebook prepares the **camera-derived traffic and visual-reliability caches** and trains the main forecasting baselines used by the SmartTraffic study.

**Important:** use this notebook for data preparation, YOLO11s + ByteTrack extraction, controlled degradation generation, and baseline checkpoint training. For the final paper tables, paired robustness protocol, statistical tests, and manuscript figures, run `02_reproduce_main_paper_results.ipynb` afterward.

Pipeline: `CCTV → YOLO11s → ByteTrack → 5-s traffic/reliability sequences → forecasting checkpoints`.

Default paper setup: 5-s bins, 300-s history, and four direct 30-s forecast blocks at 30/60/90/120 s.

In [ ]:
%pip -q install ultralytics gdown pyarrow openpyxl statsmodels scipy scikit-learn tqdm tabulate

from google.colab import drive
drive.mount('/content/drive')

import os, sys, math, json, time, random, shutil, zipfile, subprocess, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")
print("Google Drive mounted.")

In [ ]:
# ========================= CONFIG =========================
CFG = {
    "SEED": 42,
    "PROJECT_NAME": "SmartTraffic_VRA_TFormer_IJIES",

    # Main dataset — AI City Challenge 2021 Track 1
    "DOWNLOAD_AICITY": True,
    "AICITY_GDRIVE_FILE_ID": "1xGaOlCwz7Zn7SbDZiWsfG0m2UGs7eqpO",
    "AICITY_ARCHIVE_NAME": "AICity21_Track1_Vehicle_Counting.zip",

    # External perception validation — UA-DETRAC
    "RUN_DETRAC": False,
    "DETRAC_MAX_SEQUENCES": 12,
    "DETRAC_FRAME_STRIDE": 10,

    # Vision pipeline
    "YOLO_WEIGHTS": "yolo11s.pt",
    "YOLO_CONF": 0.25,
    "YOLO_IOU": 0.50,
    "VEHICLE_CLASS_IDS": [2, 3, 5, 7],   # car, motorcycle, bus, truck in COCO
    "VIDEO_FRAME_STRIDE": 2,              # ~5 FPS for typical 10 FPS AI City video
    "IMAGE_SIZE": 960,

    # Time series
    "BIN_SEC": 5,
    "LOOKBACK_STEPS": 60,                 # 5 min
    "FORECAST_BLOCK_STEPS": 6,            # 30 sec
    "N_HORIZONS": 4,
    "TRAIN_FRAC": 0.60,
    "VAL_FRAC": 0.20,

    # Controlled degradation
    "RUN_DEGRADATION": True,
    "DEGRADATION_VIDEO_LIMIT": 3,
    "DEGRADATIONS": {
        "blur": [1,3],
        "lowlight": [1,3],
        "occlusion": [1,3],
        "frame_drop": [1,3],
    },

    # Forecast model
    "BATCH_SIZE": 64,
    "EPOCHS": 80,
    "PATIENCE": 12,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "D_MODEL": 64,
    "N_HEAD": 4,
    "N_LAYERS": 2,
    "DROPOUT": 0.10,

    "FIG_DPI": 400,
}
print(json.dumps(CFG, indent=2))

In [ ]:
# ========================= SETUP =========================
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import platform, sklearn, scipy

SEED = CFG["SEED"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path("/content/drive/MyDrive") / CFG["PROJECT_NAME"]
D = {
    "root": ROOT,
    "data": ROOT/"data",
    "aicity": ROOT/"data"/"aicity21",
    "detrac": ROOT/"data"/"ua_detrac",
    "cache": ROOT/"cache",
    "clean": ROOT/"cache"/"aicity_clean",
    "degraded": ROOT/"cache"/"aicity_degraded",
    "models": ROOT/"models",
    "tables": ROOT/"tables",
    "figures": ROOT/"figures",
    "logs": ROOT/"logs",
}
for p in D.values(): p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman","Liberation Serif","DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

env = {
    "python": sys.version,
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "scipy": scipy.__version__,
    "cuda": torch.version.cuda,
    "device": str(DEVICE),
}
(D["logs"]/"environment.json").write_text(json.dumps(env,indent=2))
print("Project:", ROOT)

## Public data used

**AI City Challenge 2021 Track 1** is the main camera-to-forecast dataset.  
The official public page currently provides a Google Drive archive for the 2021 multi-class multi-movement vehicle-counting data.

**UA-DETRAC** is optional external validation for perception reliability. It provides traffic-camera imagery, vehicle boxes/types, illumination conditions, and occlusion-related annotations.

If an upstream host is temporarily unavailable, place the downloaded ZIP in the Drive path created by this notebook and rerun; existing archives are reused.

UA-DETRAC code is retained as an optional perception diagnostic but is disabled in the paper configuration.

In [ ]:
# ========================= ROBUST DOWNLOAD / EXTRACT =========================
import gdown
import zipfile
import os

def is_valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)

def safe_remove(path):
    path = Path(path)
    if path.exists():
        try:
            path.unlink()
            print("Removed invalid file:", path)
        except Exception as e:
            print("Could not remove", path, ":", e)

def unzip_once(zpath, outdir, marker):
    zpath = Path(zpath)
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    mark = outdir / marker

    if mark.exists():
        print("[cache]", zpath.name, "already extracted")
        return True

    if not is_valid_zip(zpath):
        print("[ERROR] Not a valid ZIP:", zpath)
        return False

    print("Extracting:", zpath.name)
    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            bad_member = zf.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(f"Corrupt member: {bad_member}")
            zf.extractall(outdir)
        mark.write_text("ok")
        print("Extracted to:", outdir)
        return True
    except Exception as e:
        print("[ERROR] Extraction failed:", e)
        return False

def gdrive_zip(file_id, dest, label):
    """
    Download a public Google Drive ZIP and verify its integrity.
    If a previous invalid HTML/error file exists, delete it first.
    """
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    if is_valid_zip(dest):
        print("[cache]", label, "valid ZIP already exists:", dest.name)
        return True

    if dest.exists():
        print("[repair]", label, "existing file is not a valid ZIP.")
        safe_remove(dest)

    url = f"https://drive.google.com/uc?id={file_id}"
    print("Downloading", label, "from Google Drive...")
    try:
        out = gdown.download(url, str(dest), quiet=False, fuzzy=True)
    except Exception as e:
        print("[ERROR]", label, "download failed:", e)
        return False

    if not is_valid_zip(dest):
        print("[ERROR]", label, "download completed but content is not a valid ZIP.")
        if dest.exists():
            print("Downloaded bytes:", dest.stat().st_size)
        safe_remove(dest)
        return False

    print("[OK]", label, "ZIP validated:", dest)
    return True

# ------------------------------------------------------------------
# AI CITY CHALLENGE 2021 TRACK 1 — MAIN DATASET
# ------------------------------------------------------------------
if CFG["DOWNLOAD_AICITY"]:
    aicity_zip = D["aicity"] / CFG["AICITY_ARCHIVE_NAME"]

    # Preserve already downloaded 5.83 GB archive when valid.
    if not is_valid_zip(aicity_zip):
        if aicity_zip.exists():
            print("Existing AI City archive is invalid; redownloading.")
            safe_remove(aicity_zip)

        ok = gdrive_zip(
            CFG["AICITY_GDRIVE_FILE_ID"],
            aicity_zip,
            "AI City 2021 Track 1"
        )
        if not ok:
            raise RuntimeError(
                "AI City download failed. This is the MAIN dataset, so execution stops."
            )
    else:
        print("[cache] AI City archive is valid:", aicity_zip)

    if not unzip_once(aicity_zip, D["aicity"] / "extracted", ".done"):
        raise RuntimeError("AI City archive could not be extracted.")

# ------------------------------------------------------------------
# UA-DETRAC — OPTIONAL EXTERNAL VALIDATION
#
# IMPORTANT:
# A legacy DETRAC endpoint may return HTML/error
# content while still creating a file named .zip.
# Current official UA-DETRAC page links DETRAC-Images.zip on Google
# Drive. Official public file ID:
#   1fQC6CEWOeL9pnJ4ZdD7U2gromcnWDRgm
# ------------------------------------------------------------------
DETRAC_READY = False

if CFG["RUN_DETRAC"]:
    # Remove the obsolete/broken file from notebook v1 if present.
    old_bad = D["detrac"] / "DETRAC-train-data.zip"
    if old_bad.exists() and not is_valid_zip(old_bad):
        print("Found broken legacy DETRAC-train-data.zip from v1.")
        safe_remove(old_bad)

    detrac_images_zip = D["detrac"] / "DETRAC-Images.zip"
    detrac_xml_zip = D["detrac"] / "DETRAC-Train_Annotations-XML.zip"

    # Current official DETRAC Images link.
    images_ok = gdrive_zip(
        "1fQC6CEWOeL9pnJ4ZdD7U2gromcnWDRgm",
        detrac_images_zip,
        "UA-DETRAC Images"
    )

    # Existing public XML annotation mirror/file used by many UA-DETRAC setups.
    # Validate because a Google Drive error page can also be saved as .zip.
    xml_ok = gdrive_zip(
        "12xJc8S0Z7lYaAadsi2CoSK3WqH2OkUBu",
        detrac_xml_zip,
        "UA-DETRAC XML annotations"
    )

    if images_ok and xml_ok:
        img_extract_ok = unzip_once(
            detrac_images_zip, D["detrac"] / "images", ".images_done"
        )
        xml_extract_ok = unzip_once(
            detrac_xml_zip, D["detrac"] / "xml", ".xml_done"
        )
        DETRAC_READY = bool(img_extract_ok and xml_extract_ok)
    else:
        DETRAC_READY = False

    if not DETRAC_READY:
        print("\n" + "=" * 72)
        print("WARNING: UA-DETRAC is unavailable, so EXTERNAL VALIDATION is skipped.")
        print("The MAIN AI City → VRA-TFormer forecasting pipeline will continue.")
        print("You can rerun this cell later when DETRAC is available.")
        print("=" * 72 + "\n")
        CFG["RUN_DETRAC"] = False

print("Dataset acquisition stage complete.")
print("AI City ready:", (D["aicity"] / "extracted" / ".done").exists())
print("UA-DETRAC ready:", DETRAC_READY if "DETRAC_READY" in globals() else False)

In [ ]:
# ========================= DISCOVER AI CITY VIDEOS =========================
VIDEO_EXTS = {".mp4",".avi",".mov",".mkv",".m4v"}
aicity_root = D["aicity"]/"extracted"
AICITY_VIDEOS = sorted([
    p for p in aicity_root.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS
])
print("AI City videos:", len(AICITY_VIDEOS))
for p in AICITY_VIDEOS[:10]:
    print(" -", p.relative_to(aicity_root))

if not AICITY_VIDEOS:
    raise RuntimeError("No AI City videos found. Inspect the extracted archive structure in Drive.")

inventory = pd.DataFrame({
    "sequence":[p.stem for p in AICITY_VIDEOS],
    "path":[str(p) for p in AICITY_VIDEOS],
    "size_mb":[p.stat().st_size/(1024**2) for p in AICITY_VIDEOS],
})
inventory.to_csv(D["tables"]/"aicity_video_inventory.csv",index=False)
display(inventory.head())

In [ ]:
# ========================= YOLO + BYTETRACK EXTRACTION =========================
import cv2
from ultralytics import YOLO

COCO_ID_TO_NAME = {2:"car",3:"motorcycle",5:"bus",7:"truck"}
TRAFFIC_COLS = ["car","motorcycle","bus","truck","total"]
REL_COLS = [
    "mean_conf","track_continuity","blur_log",
    "illumination_quality","occlusion_quality","frame_availability"
]

def iou_matrix(boxes):
    b=np.asarray(boxes,float); n=len(b)
    if n<2: return np.zeros((n,n))
    x1=np.maximum(b[:,None,0],b[None,:,0]); y1=np.maximum(b[:,None,1],b[None,:,1])
    x2=np.minimum(b[:,None,2],b[None,:,2]); y2=np.minimum(b[:,None,3],b[None,:,3])
    inter=np.clip(x2-x1,0,None)*np.clip(y2-y1,0,None)
    area=np.clip(b[:,2]-b[:,0],0,None)*np.clip(b[:,3]-b[:,1],0,None)
    u=area[:,None]+area[None,:]-inter+1e-9
    m=inter/u; np.fill_diagonal(m,0)
    return m

def frame_quality(frame, boxes, confs, ids, prev_ids):
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    bright=float(gray.mean()/255)
    illum=float(np.clip(1-abs(bright-.5)/.5,0,1))
    blur_log=float(np.log1p(cv2.Laplacian(gray,cv2.CV_64F).var()))
    mean_conf=float(np.mean(confs)) if len(confs) else np.nan

    ids=set(int(i) for i in ids if i is not None and int(i)>=0)
    prev=set(int(i) for i in prev_ids if i is not None and int(i)>=0)
    cont=(len(ids&prev)/len(ids)) if ids else np.nan

    if len(boxes)>=2:
        overlap=iou_matrix(boxes).max(axis=1)
        occ_risk=float(np.mean(overlap>.10))
    else:
        occ_risk=0.0
    return {
        "mean_conf":mean_conf,
        "track_continuity":cont,
        "blur_log":blur_log,
        "brightness":bright,
        "illumination_quality":illum,
        "occlusion_quality":1-occ_risk,
    }

def degrade(frame, condition="clean", severity=0, seed=42):
    if condition=="clean" or severity==0: return frame
    out=frame.copy(); h,w=out.shape[:2]
    rng=np.random.default_rng(seed)
    if condition=="blur":
        k={1:5,2:11,3:21}[severity]
        out=cv2.GaussianBlur(out,(k,k),0)
    elif condition=="lowlight":
        f={1:.70,2:.45,3:.25}[severity]
        out=np.clip(out.astype(np.float32)*f,0,255).astype(np.uint8)
    elif condition=="jpeg":
        q={1:60,2:35,3:15}[severity]
        ok,enc=cv2.imencode(".jpg",out,[int(cv2.IMWRITE_JPEG_QUALITY),q])
        if ok: out=cv2.imdecode(enc,cv2.IMREAD_COLOR)
    elif condition=="occlusion":
        n={1:1,2:2,3:3}[severity]
        frac={1:.10,2:.16,3:.22}[severity]
        for _ in range(n):
            rw,rh=max(8,int(w*frac)),max(8,int(h*frac))
            x=int(rng.integers(0,max(1,w-rw)))
            y=int(rng.integers(int(h*.25),max(int(h*.25)+1,h-rh)))
            cv2.rectangle(out,(x,y),(x+rw,y+rh),(0,0,0),-1)
    return out

def process_video(video_path, model, condition="clean", severity=0):
    # Predictor reset avoids tracker IDs leaking across videos.
    model.predictor=None
    cap=cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): raise RuntimeError(f"Cannot open {video_path}")

    fps=float(cap.get(cv2.CAP_PROP_FPS) or 10)
    n_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration=n_frames/fps if n_frames else 0
    stride=CFG["VIDEO_FRAME_STRIDE"]; bin_sec=CFG["BIN_SEC"]

    seen=set(); prev=set(); events=[]; frames=[]
    fidx=-1
    bar=tqdm(total=n_frames or None,desc=f"{video_path.stem}:{condition}-{severity}",leave=False)

    while True:
        ok,frame=cap.read()
        if not ok: break
        fidx+=1
        if bar.total: bar.update(1)
        if fidx%stride: continue

        t=fidx/fps; bidx=int(t//bin_sec)

        if condition=="frame_drop" and severity:
            p={1:.10,2:.30,3:.50}[severity]
            if np.random.default_rng(SEED+fidx).random()<p:
                continue

        proc=degrade(frame,condition,severity,SEED+fidx+1000*severity)
        r=model.track(
            proc,persist=True,tracker="bytetrack.yaml",
            classes=CFG["VEHICLE_CLASS_IDS"],
            conf=CFG["YOLO_CONF"],iou=CFG["YOLO_IOU"],
            imgsz=CFG["IMAGE_SIZE"],
            device=0 if torch.cuda.is_available() else "cpu",
            verbose=False
        )[0]

        if r.boxes is not None and len(r.boxes):
            boxes=r.boxes.xyxy.detach().cpu().numpy()
            conf=r.boxes.conf.detach().cpu().numpy()
            cls=r.boxes.cls.detach().cpu().numpy().astype(int)
            ids=(r.boxes.id.detach().cpu().numpy().astype(int)
                 if r.boxes.id is not None else np.full(len(boxes),-1))
        else:
            boxes=np.empty((0,4)); conf=np.array([]); cls=np.array([],int); ids=np.array([],int)

        q=frame_quality(proc,boxes,conf,ids,prev)
        q.update({"bin_idx":bidx,"frame_idx":fidx,"n_det":len(boxes)})
        frames.append(q)

        cur=set()
        for tid,cid,cf in zip(ids,cls,conf):
            if tid<0: continue
            cur.add(int(tid))
            if int(tid) not in seen:
                seen.add(int(tid))
                events.append({
                    "bin_idx":bidx,
                    "vehicle_class":COCO_ID_TO_NAME.get(int(cid),"other"),
                    "confidence":float(cf),
                })
        prev=cur

    bar.close(); cap.release()

    fdf=pd.DataFrame(frames); edf=pd.DataFrame(events)
    n_bins=int(math.ceil(duration/bin_sec)) if duration else (int(fdf.bin_idx.max())+1 if len(fdf) else 0)
    base=pd.DataFrame({"bin_idx":np.arange(n_bins)})

    if len(edf):
        cnt=edf.groupby(["bin_idx","vehicle_class"]).size().unstack(fill_value=0).reset_index()
    else:
        cnt=pd.DataFrame({"bin_idx":[]})
    for c in ["car","motorcycle","bus","truck"]:
        if c not in cnt.columns: cnt[c]=0

    if len(fdf):
        qdf=fdf.groupby("bin_idx").agg(
            mean_conf=("mean_conf","mean"),
            track_continuity=("track_continuity","mean"),
            blur_log=("blur_log","mean"),
            brightness=("brightness","mean"),
            illumination_quality=("illumination_quality","mean"),
            occlusion_quality=("occlusion_quality","mean"),
            sampled_frames=("frame_idx","count"),
            mean_n_det=("n_det","mean")
        ).reset_index()
    else:
        qdf=pd.DataFrame({"bin_idx":[]})

    out=base.merge(cnt,on="bin_idx",how="left").merge(qdf,on="bin_idx",how="left")
    for c in ["car","motorcycle","bus","truck"]:
        out[c]=out[c].fillna(0).astype(float)
    out["total"]=out[["car","motorcycle","bus","truck"]].sum(axis=1)
    expected=max(1e-6,bin_sec*fps/stride)
    out["frame_availability"]=(out["sampled_frames"].fillna(0)/expected).clip(0,1)
    out["sequence"]=video_path.stem
    out["condition"]=condition; out["severity"]=severity
    out["t_start_s"]=out["bin_idx"]*bin_sec
    out["fps"]=fps; out["duration_s"]=duration
    return out

detector=YOLO(CFG["YOLO_WEIGHTS"])

In [ ]:
# ========================= CLEAN EXTRACTION — CACHED =========================
clean_files=[]
for vp in tqdm(AICITY_VIDEOS,desc="Clean sequences"):
    op=D["clean"]/f"{vp.stem}.parquet"
    clean_files.append(op)
    if not op.exists():
        process_video(vp,detector,"clean",0).to_parquet(op,index=False)

clean_df=pd.concat([pd.read_parquet(p) for p in clean_files],ignore_index=True)
clean_df.to_parquet(D["cache"]/"aicity_clean_all.parquet",index=False)
print("Sequences:",clean_df.sequence.nunique(),"bins:",len(clean_df))
display(clean_df.head())

In [ ]:
# ========================= CONTROLLED DEGRADATION — CACHED =========================
degraded_df=pd.DataFrame()
if CFG["RUN_DEGRADATION"]:
    n=min(CFG["DEGRADATION_VIDEO_LIMIT"],len(AICITY_VIDEOS))
    ids=np.linspace(0,len(AICITY_VIDEOS)-1,n,dtype=int)
    subset=[AICITY_VIDEOS[i] for i in ids]
    print("Robustness subset:",[x.name for x in subset])

    files=[]
    for cond,sevs in CFG["DEGRADATIONS"].items():
        for sev in sevs:
            od=D["degraded"]/cond/f"severity_{sev}"
            od.mkdir(parents=True,exist_ok=True)
            for vp in subset:
                op=od/f"{vp.stem}.parquet"; files.append(op)
                if not op.exists():
                    process_video(vp,detector,cond,sev).to_parquet(op,index=False)

    degraded_df=pd.concat([pd.read_parquet(p) for p in files],ignore_index=True)
    degraded_df.to_parquet(D["cache"]/"aicity_degraded_all.parquet",index=False)
    print("Degraded bins:",len(degraded_df))

In [ ]:
# ========================= DATASET TABLE + FIGURES 1–2 =========================
def savefig(fig,name):
    fig.savefig(D["figures"]/f"{name}.png",dpi=CFG["FIG_DPI"],bbox_inches="tight")
    fig.savefig(D["figures"]/f"{name}.pdf",bbox_inches="tight")

rows=[]
for seq,g in clean_df.groupby("sequence"):
    rows.append({
        "sequence":seq,"bins":len(g),
        "duration_min":g.duration_s.iloc[0]/60,
        "vehicles_counted":g.total.sum(),
        "car":g.car.sum(),"motorcycle":g.motorcycle.sum(),
        "bus":g.bus.sum(),"truck":g.truck.sum(),
        "mean_conf":g.mean_conf.mean(),
        "frame_availability":g.frame_availability.mean(),
    })
dataset_summary=pd.DataFrame(rows)
dataset_summary.to_csv(D["tables"]/"table_dataset_sequence_summary.csv",index=False)
display(dataset_summary)

# Fig 1 — architecture
fig,ax=plt.subplots(figsize=(10.5,2.8)); ax.axis("off")
boxes=[
(.02,.40,.12,.28,"CCTV\nVideo"),(.18,.40,.13,.28,"YOLO +\nByteTrack"),
(.36,.60,.16,.25,"Traffic\nObservation"),(.36,.18,.16,.25,"Visual\nReliability"),
(.58,.40,.16,.28,"Reliability-\nAware Fusion"),(.79,.40,.16,.28,"VRA-TFormer\nForecast")]
for x,y,w,h,t in boxes:
    ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,lw=1.4))
    ax.text(x+w/2,y+h/2,t,ha="center",va="center")
for a,b in [((.14,.54),(.18,.54)),((.31,.54),(.36,.72)),((.31,.54),(.36,.30)),
            ((.52,.72),(.58,.57)),((.52,.30),(.58,.51)),((.74,.54),(.79,.54))]:
    ax.annotate("",xy=b,xytext=a,arrowprops=dict(arrowstyle="->",lw=1.2))
ax.text(.66,.10,"Feature gating + reliability-biased temporal attention",ha="center",fontsize=9)
savefig(fig,"Fig1_VRATFormer_architecture"); plt.show()

# Fig 2 — example time series
seq0=dataset_summary.sort_values("bins",ascending=False).iloc[0].sequence
g=clean_df[clean_df.sequence==seq0]
fig,ax=plt.subplots(figsize=(8,3.5))
ax.plot(g.t_start_s/60,g.total,lw=1.2)
ax.set_xlabel("Time (min)"); ax.set_ylabel("Vehicle count / 5 s")
a2=ax.twinx()
a2.plot(g.t_start_s/60,g.mean_conf.interpolate(limit_direction="both"),ls="--",lw=1)
a2.set_ylabel("Mean detection confidence")
ax.set_title(f"Camera-derived traffic observation — {seq0}")
fig.tight_layout(); savefig(fig,"Fig2_traffic_confidence_series"); plt.show()

In [ ]:
# ========================= UA-DETRAC EXTERNAL VALIDATION =========================
import xml.etree.ElementTree as ET

def detrac_maps():
    imgs=[p for p in (D["detrac"]/"images").rglob("*") if p.is_dir() and p.name.startswith("MVI_")]
    xmls=list((D["detrac"]/"xml").rglob("*.xml"))
    im={p.name:p for p in imgs}; xm={p.stem:p for p in xmls}
    keys=sorted(set(im)&set(xm))
    return im,xm,keys

def parse_xml(path):
    root=ET.parse(path).getroot()
    weather="unknown"
    for e in root.iter():
        for k,v in e.attrib.items():
            if k.lower() in {"weather","illumination"}:
                weather=v; break
        if weather!="unknown": break
    frames={}
    for fr in root.findall(".//frame"):
        num=int(fr.attrib.get("num","0")); bb=[]
        for t in fr.findall(".//target"):
            b=t.find("box")
            if b is None: continue
            l=float(b.attrib.get("left",0)); top=float(b.attrib.get("top",0))
            w=float(b.attrib.get("width",0)); h=float(b.attrib.get("height",0))
            bb.append([l,top,l+w,top+h])
        frames[num]=bb
    return weather,frames

def iou1(box,boxes):
    if len(boxes)==0:return np.array([])
    b=np.asarray(box,float); B=np.asarray(boxes,float)
    x1=np.maximum(b[0],B[:,0]); y1=np.maximum(b[1],B[:,1])
    x2=np.minimum(b[2],B[:,2]); y2=np.minimum(b[3],B[:,3])
    inter=np.clip(x2-x1,0,None)*np.clip(y2-y1,0,None)
    a=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    A=np.clip(B[:,2]-B[:,0],0,None)*np.clip(B[:,3]-B[:,1],0,None)
    return inter/(a+A-inter+1e-9)

def ap101(records,n_gt):
    if not records or n_gt==0:return np.nan
    recs=sorted(records,key=lambda x:-x[0])
    tp=np.array([x[1] for x in recs],float); fp=1-tp
    tp=np.cumsum(tp); fp=np.cumsum(fp)
    recall=tp/max(n_gt,1); precision=tp/np.maximum(tp+fp,1e-9)
    return sum((precision[recall>=r].max() if np.any(recall>=r) else 0)/101 for r in np.linspace(0,1,101))

def eval_detrac(model):
    im,xm,keys=detrac_maps()
    keys=keys[:CFG["DETRAC_MAX_SEQUENCES"]]
    if not keys:
        print("No matched DETRAC image/XML sequences found.")
        return pd.DataFrame(),pd.DataFrame()

    thrs=np.arange(.50,1.00,.05); records={float(t):[] for t in thrs}
    n_gt=0; stats=[]
    for seq in tqdm(keys,desc="UA-DETRAC"):
        weather,gtframes=parse_xml(xm[seq])
        imgs=sorted(list(im[seq].glob("*.jpg"))+list(im[seq].glob("*.png")))
        for ip in imgs[::CFG["DETRAC_FRAME_STRIDE"]]:
            nums=re.findall(r"(\d+)",ip.stem)
            if not nums:continue
            fn=int(nums[-1]); gt=gtframes.get(fn,[]); n_gt+=len(gt)
            frame=cv2.imread(str(ip))
            if frame is None:continue

            r=model.predict(frame,classes=CFG["VEHICLE_CLASS_IDS"],conf=.001,
                            iou=CFG["YOLO_IOU"],imgsz=CFG["IMAGE_SIZE"],
                            device=0 if torch.cuda.is_available() else "cpu",verbose=False)[0]
            if r.boxes is not None and len(r.boxes):
                pb=r.boxes.xyxy.detach().cpu().numpy()
                pc=r.boxes.conf.detach().cpu().numpy()
            else:
                pb=np.empty((0,4)); pc=np.array([])

            q=frame_quality(frame,pb,pc,[],[])
            pred_count=int(np.sum(pc>=CFG["YOLO_CONF"]))
            proxy=np.nanmean([
                q["mean_conf"] if np.isfinite(q["mean_conf"]) else .5,
                q["illumination_quality"],q["occlusion_quality"]
            ])
            stats.append({
                "sequence":seq,"weather":weather,"frame":fn,
                "gt_count":len(gt),"pred_count":pred_count,
                "abs_count_error":abs(pred_count-len(gt)),
                "mean_conf":q["mean_conf"],
                "illumination_quality":q["illumination_quality"],
                "occlusion_quality":q["occlusion_quality"],
                "quality_proxy":proxy
            })

            order=np.argsort(-pc); pb=pb[order]; pc=pc[order]
            for thr in thrs:
                used=set()
                for box,cf in zip(pb,pc):
                    iv=iou1(box,gt); matched=0
                    if len(iv):
                        for j in np.argsort(-iv):
                            if iv[j]>=thr and int(j) not in used:
                                used.add(int(j)); matched=1; break
                    records[float(thr)].append((float(cf),matched))

    ap=pd.DataFrame([{"IoU":float(t),"AP":ap101(records[float(t)],n_gt)} for t in thrs])
    fr=pd.DataFrame(stats)
    if len(fr):
        summ=pd.DataFrame([{
            "AP50":ap.loc[np.isclose(ap.IoU,.5),"AP"].iloc[0],
            "mAP50_95":ap.AP.mean(),
            "Count_MAE":fr.abs_count_error.mean(),
            "Count_WAPE_%":fr.abs_count_error.sum()/max(fr.gt_count.sum(),1)*100,
            "N_frames":len(fr),"N_GT_objects":fr.gt_count.sum()
        }])
        summ.to_csv(D["tables"]/"table_detrac_perception_summary.csv",index=False)
        ap.to_csv(D["tables"]/"detrac_ap.csv",index=False)
        fr.to_csv(D["tables"]/"detrac_frame_results.csv",index=False)
        display(summ)

        fig,ax=plt.subplots(figsize=(5.2,3.6))
        x=fr.quality_proxy.fillna(fr.quality_proxy.median()); y=fr.abs_count_error
        ax.scatter(x,y,s=12,alpha=.45)
        if len(fr)>=3:
            z=np.polyfit(x,y,1); xx=np.linspace(x.min(),x.max(),100)
            ax.plot(xx,np.polyval(z,xx),lw=1.2)
        ax.set_xlabel("Visual quality proxy"); ax.set_ylabel("Absolute count error")
        ax.set_title("External perception validation — UA-DETRAC")
        fig.tight_layout(); savefig(fig,"Fig3_DETRAC_quality_count_error"); plt.show()
    return ap,fr

if CFG["RUN_DETRAC"] and globals().get("DETRAC_READY", False):
    detrac_ap,detrac_frames=eval_detrac(detector)
else:
    print("UA-DETRAC external validation skipped; main experiment continues.")
    detrac_ap,detrac_frames=pd.DataFrame(),pd.DataFrame()

In [ ]:
# ========================= FORECAST DATA PREPARATION =========================
from sklearn.preprocessing import StandardScaler

clean_df=pd.read_parquet(D["cache"]/"aicity_clean_all.parquet").sort_values(["sequence","bin_idx"]).reset_index(drop=True)

# Interpolate descriptors only inside each camera sequence.
for c in REL_COLS:
    clean_df[c]=clean_df.groupby("sequence")[c].transform(lambda s:s.interpolate(limit_direction="both"))

bounds={}
for seq,g in clean_df.groupby("sequence"):
    n=len(g); tr=int(n*CFG["TRAIN_FRAC"]); va=int(n*(CFG["TRAIN_FRAC"]+CFG["VAL_FRAC"]))
    bounds[seq]=(tr,va,n)

train_row=np.zeros(len(clean_df),bool)
for seq,idx in clean_df.groupby("sequence").groups.items():
    tr=bounds[seq][0]; ids=list(idx); train_row[ids[:tr]]=True

med=clean_df.loc[train_row,REL_COLS].median()
for c in REL_COLS: clean_df[c]=clean_df[c].fillna(med[c])

xs=StandardScaler().fit(clean_df.loc[train_row,TRAFFIC_COLS])
qs=StandardScaler().fit(clean_df.loc[train_row,REL_COLS])

L=CFG["LOOKBACK_STEPS"]; B=CFG["FORECAST_BLOCK_STEPS"]; H=CFG["N_HORIZONS"]
samples=[]
for seq,g in clean_df.groupby("sequence",sort=False):
    g=g.reset_index(drop=True); tr,va,n=bounds[seq]
    X=xs.transform(g[TRAFFIC_COLS]).astype(np.float32)
    Q=qs.transform(g[REL_COLS]).astype(np.float32)
    yr=g.total.values.astype(float)
    future=B*H
    for t in range(L,n-future+1):
        y=np.array([yr[t+h*B:t+(h+1)*B].sum() for h in range(H)],np.float32)
        end=t+future
        if end<=tr: split="train"
        elif t>=tr and end<=va: split="val"
        elif t>=va: split="test"
        else: continue
        samples.append({"sequence":seq,"split":split,"t":t,"x":X[t-L:t],"q":Q[t-L:t],"y_raw":y})

ytr=np.vstack([r["y_raw"] for r in samples if r["split"]=="train"])
ys=StandardScaler().fit(ytr.reshape(-1,1))
for r in samples:
    r["y"]=ys.transform(r["y_raw"].reshape(-1,1)).reshape(-1).astype(np.float32)

scaler_meta={
    "traffic_cols":TRAFFIC_COLS,"reliability_cols":REL_COLS,
    "traffic_mean":xs.mean_.tolist(),"traffic_scale":xs.scale_.tolist(),
    "reliability_mean":qs.mean_.tolist(),"reliability_scale":qs.scale_.tolist(),
    "target_mean":ys.mean_.tolist(),"target_scale":ys.scale_.tolist()
}
(D["models"]/"scalers.json").write_text(json.dumps(scaler_meta,indent=2))

split_summary=pd.DataFrame({
    "split":["train","val","test"],
    "windows":[sum(r["split"]==s for r in samples) for s in ["train","val","test"]]
})
split_summary.to_csv(D["tables"]/"table_forecasting_split_summary.csv",index=False)
display(split_summary)

In [ ]:
# ========================= DATALOADERS + MODELS =========================
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.nn.functional as F

class WinDS(Dataset):
    def __init__(self,rows,split): self.rows=[r for r in rows if r["split"]==split]
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        return (torch.tensor(r["x"]),torch.tensor(r["q"]),torch.tensor(r["y"]),r["sequence"],int(r["t"]))

ds={s:WinDS(samples,s) for s in ["train","val","test"]}
ld={
    "train":DataLoader(ds["train"],batch_size=CFG["BATCH_SIZE"],shuffle=True,num_workers=0),
    "val":DataLoader(ds["val"],batch_size=CFG["BATCH_SIZE"],shuffle=False,num_workers=0),
    "test":DataLoader(ds["test"],batch_size=CFG["BATCH_SIZE"],shuffle=False,num_workers=0)
}
print({k:len(v.dataset) for k,v in ld.items()})

NT=len(TRAFFIC_COLS); NR=len(REL_COLS); NO=CFG["N_HORIZONS"]; DM=CFG["D_MODEL"]

class Pos(nn.Module):
    def __init__(self,d,maxlen=2048):
        super().__init__(); pe=torch.zeros(maxlen,d); p=torch.arange(maxlen).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,d,2).float()*(-math.log(10000)/d))
        pe[:,0::2]=torch.sin(p*div); pe[:,1::2]=torch.cos(p*div)
        self.register_buffer("pe",pe.unsqueeze(0),persistent=False)
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__(); self.r=nn.LSTM(NT,DM,2,batch_first=True,dropout=CFG["DROPOUT"]); self.h=nn.Linear(DM,NO)
    def forward(self,x,q=None): z,_=self.r(x); return self.h(z[:,-1])

class GRUModel(nn.Module):
    def __init__(self):
        super().__init__(); self.r=nn.GRU(NT,DM,2,batch_first=True,dropout=CFG["DROPOUT"]); self.h=nn.Linear(DM,NO)
    def forward(self,x,q=None): z,_=self.r(x); return self.h(z[:,-1])

class TransformerModel(nn.Module):
    def __init__(self,indim=NT):
        super().__init__(); self.p=nn.Linear(indim,DM); self.pos=Pos(DM)
        lay=nn.TransformerEncoderLayer(DM,CFG["N_HEAD"],DM*4,CFG["DROPOUT"],batch_first=True,norm_first=True)
        self.e=nn.TransformerEncoder(lay,CFG["N_LAYERS"]); self.h=nn.Sequential(nn.LayerNorm(DM),nn.Linear(DM,NO))
    def forward(self,x,q=None): z=self.e(self.pos(self.p(x))); return self.h(z.mean(1))

class ConcatModel(nn.Module):
    def __init__(self): super().__init__(); self.m=TransformerModel(NT+NR)
    def forward(self,x,q=None): return self.m(torch.cat([x,q],-1))

class PatchTSTModel(nn.Module):
    # Compact channel-independent PatchTST-style comparator.
    def __init__(self,patch=12,stride=6):
        super().__init__(); self.patch=patch; self.stride=stride
        P=1+(CFG["LOOKBACK_STEPS"]-patch)//stride
        self.pp=nn.Linear(patch,DM); self.pos=nn.Parameter(torch.zeros(1,P,DM))
        lay=nn.TransformerEncoderLayer(DM,CFG["N_HEAD"],DM*4,CFG["DROPOUT"],batch_first=True,norm_first=True)
        self.e=nn.TransformerEncoder(lay,CFG["N_LAYERS"])
        self.h=nn.Linear(NT*DM,NO)
    def forward(self,x,q=None):
        p=x.transpose(1,2).unfold(2,self.patch,self.stride); B,C,P,PL=p.shape
        z=self.pp(p.reshape(B*C,P,PL))+self.pos[:,:P]
        z=self.e(z).mean(1).reshape(B,C*DM); return self.h(z)

class ITransformerModel(nn.Module):
    # Variates are tokens; each token is projected from its full temporal history.
    def __init__(self):
        super().__init__(); self.p=nn.Linear(CFG["LOOKBACK_STEPS"],DM)
        self.ve=nn.Parameter(torch.zeros(1,NT,DM))
        lay=nn.TransformerEncoderLayer(DM,CFG["N_HEAD"],DM*4,CFG["DROPOUT"],batch_first=True,norm_first=True)
        self.e=nn.TransformerEncoder(lay,CFG["N_LAYERS"]); self.h=nn.Linear(NT*DM,NO)
    def forward(self,x,q=None):
        z=self.e(self.p(x.transpose(1,2))+self.ve).reshape(x.size(0),-1)
        return self.h(z)

class RelLayer(nn.Module):
    def __init__(self):
        super().__init__(); self.nh=CFG["N_HEAD"]
        self.att=nn.MultiheadAttention(DM,self.nh,dropout=CFG["DROPOUT"],batch_first=True)
        self.n1=nn.LayerNorm(DM); self.n2=nn.LayerNorm(DM)
        self.ff=nn.Sequential(nn.Linear(DM,DM*4),nn.GELU(),nn.Dropout(CFG["DROPOUT"]),nn.Linear(DM*4,DM))
    def forward(self,x,r=None,lam=None):
        mask=None
        if r is not None and lam is not None:
            B,L,_=r.shape
            keybias=lam*torch.log(r.squeeze(-1).clamp_min(1e-5))
            mask=keybias[:,None,:].expand(B,L,L).repeat_interleave(self.nh,0)
        h=self.n1(x); a,_=self.att(h,h,h,attn_mask=mask,need_weights=False)
        x=x+a; return x+self.ff(self.n2(x))

class VRATFormer(nn.Module):
    def __init__(self,use_gate=True,use_rel_att=True):
        super().__init__(); self.use_gate=use_gate; self.use_rel_att=use_rel_att
        self.obs=nn.Sequential(nn.Linear(NT,DM),nn.GELU())
        self.rel=nn.Sequential(nn.Linear(NR,DM//2),nn.GELU())
        self.gate=nn.Sequential(nn.Linear(DM//2,1),nn.Sigmoid())
        self.ctx=nn.GRU(NT,DM,batch_first=True)
        self.pos=Pos(DM); self.layers=nn.ModuleList([RelLayer() for _ in range(CFG["N_LAYERS"])])
        self.rawlam=nn.Parameter(torch.tensor(0.0))
        self.head=nn.Sequential(nn.LayerNorm(DM),nn.Linear(DM,NO))
    def forward(self,x,q=None,return_reliability=False):
        o=self.obs(x); c,_=self.ctx(x); r=self.gate(self.rel(q))
        z=r*o+(1-r)*c if self.use_gate else o
        z=self.pos(z); lam=F.softplus(self.rawlam) if self.use_rel_att else None
        rr=r if self.use_rel_att else None
        for l in self.layers:z=l(z,rr,lam)
        pred=self.head(z.mean(1))
        return (pred,r,lam) if return_reliability else pred

def fresh_models():
    return {
        "LSTM":LSTMModel(),
        "GRU":GRUModel(),
        "Transformer":TransformerModel(),
        "PatchTST":PatchTSTModel(),
        "iTransformer":ITransformerModel(),
        "Transformer+ReliabilityConcat":ConcatModel(),
        "VRA-TFormer":VRATFormer(True,True),
    }

In [ ]:
# ========================= TRAIN / TEST =========================
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

def inv(a):
    a=np.asarray(a); sh=a.shape
    return ys.inverse_transform(a.reshape(-1,1)).reshape(sh)

def epoch(model,loader,opt=None):
    train=opt is not None; model.train(train); total=0
    for x,q,y,_,_ in loader:
        x,q,y=x.to(DEVICE),q.to(DEVICE),y.to(DEVICE)
        if train: opt.zero_grad(set_to_none=True)
        p=model(x,q); loss=F.mse_loss(p,y)
        if train:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        total+=loss.item()*len(x)
    return total/max(len(loader.dataset),1)

@torch.no_grad()
def pred(model,loader):
    model.eval(); yy=[]; pp=[]; ss=[]; tt=[]
    for x,q,y,s,t in loader:
        p=model(x.to(DEVICE),q.to(DEVICE)).cpu().numpy()
        yy.append(y.numpy()); pp.append(p); ss.extend(list(s)); tt.extend([int(v) for v in t])
    y=np.concatenate(yy); p=np.concatenate(pp)
    return inv(y),inv(p),ss,tt

def metrics(y,p,name):
    out=[]
    for j in range(NO):
        yt,yp=y[:,j],p[:,j]
        out.append({
            "Model":name,"Horizon":f"{30*(j+1)}s",
            "MAE":mean_absolute_error(yt,yp),
            "RMSE":mean_squared_error(yt,yp)**.5,
            "WAPE_%":np.abs(yt-yp).sum()/max(np.abs(yt).sum(),1e-9)*100,
            "R2":r2_score(yt,yp) if len(np.unique(yt))>1 else np.nan
        })
    return pd.DataFrame(out)

def train_model(name,m):
    m=m.to(DEVICE)
    safe=re.sub(r"[^A-Za-z0-9_-]+","_",name)
    ck=D["models"]/f"{safe}.pt"; hc=D["logs"]/f"{safe}_history.csv"
    if ck.exists():
        m.load_state_dict(torch.load(ck,map_location=DEVICE))
        return m,(pd.read_csv(hc) if hc.exists() else pd.DataFrame())
    opt=torch.optim.AdamW(m.parameters(),lr=CFG["LR"],weight_decay=CFG["WEIGHT_DECAY"])
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=.5,patience=4)
    best=float("inf"); wait=0; hist=[]
    for ep in range(1,CFG["EPOCHS"]+1):
        tr=epoch(m,ld["train"],opt); va=epoch(m,ld["val"]); sch.step(va)
        hist.append({"epoch":ep,"train_mse_scaled":tr,"val_mse_scaled":va,"lr":opt.param_groups[0]["lr"]})
        if va<best-1e-6:
            best=va; wait=0; torch.save(m.state_dict(),ck)
        else:
            wait+=1
            if wait>=CFG["PATIENCE"]:break
    m.load_state_dict(torch.load(ck,map_location=DEVICE))
    h=pd.DataFrame(hist); h.to_csv(hc,index=False)
    return m,h

@torch.no_grad()
def efficiency(m,loader,n=30):
    m.eval(); batches=[]
    for x,q,_,_,_ in loader:
        batches.append((x.to(DEVICE),q.to(DEVICE)))
        if len(batches)>=n:break
    for x,q in batches[:3]:m(x,q)
    if torch.cuda.is_available():torch.cuda.synchronize()
    t0=time.perf_counter(); ns=0
    for x,q in batches:m(x,q);ns+=len(x)
    if torch.cuda.is_available():torch.cuda.synchronize()
    dt=time.perf_counter()-t0
    return {
        "Params":sum(p.numel() for p in m.parameters()),
        "Latency_ms_per_batch":dt/max(len(batches),1)*1000,
        "Latency_ms_per_sample":dt/max(ns,1)*1000
    }

trained={}; histories={}; tabs=[]; eff=[]; predstore={}

# Persistence
testrows=[r for r in samples if r["split"]=="test"]
ytest=np.vstack([r["y_raw"] for r in testrows]); pp=[]
for r in testrows:
    raw=xs.inverse_transform(r["x"])
    last30=raw[-CFG["FORECAST_BLOCK_STEPS"]:,TRAFFIC_COLS.index("total")].sum()
    pp.append([last30]*NO)
pp=np.asarray(pp); tabs.append(metrics(ytest,pp,"Persistence")); predstore["Persistence"]=(ytest,pp)

for name,m in fresh_models().items():
    print("\n",name)
    m,h=train_model(name,m); trained[name]=m; histories[name]=h
    y,p,_,_=pred(m,ld["test"]); tabs.append(metrics(y,p,name)); predstore[name]=(y,p)
    e=efficiency(m,ld["test"]); e["Model"]=name; eff.append(e)

results=pd.concat(tabs,ignore_index=True)
efficiency_df=pd.DataFrame(eff)
results.to_csv(D["tables"]/"table_main_forecasting_results.csv",index=False)
efficiency_df.to_csv(D["tables"]/"table_model_efficiency.csv",index=False)
display(results); display(efficiency_df)

In [ ]:
# ========================= ABLATION =========================
spec={
    "A0_Transformer":trained["Transformer"],
    "A1_ReliabilityConcat":trained["Transformer+ReliabilityConcat"],
    "A2_ReliabilityGate":VRATFormer(True,False),
    "A3_VRA-TFormer":trained["VRA-TFormer"],
}
ab=[]; abpred={}
for name,m in spec.items():
    if name=="A2_ReliabilityGate":m,_=train_model(name,m)
    y,p,_,_=pred(m,ld["test"])
    ab.append(metrics(y,p,name)); abpred[name]=(y,p)
ablation=pd.concat(ab,ignore_index=True)
ablation.to_csv(D["tables"]/"table_ablation.csv",index=False)
display(ablation)

In [ ]:
# ========================= ROBUSTNESS =========================
def ext_windows(df):
    rows=[]; L=CFG["LOOKBACK_STEPS"]; B=CFG["FORECAST_BLOCK_STEPS"]; H=CFG["N_HORIZONS"]
    for seq,g in df.groupby("sequence",sort=False):
        g=g.sort_values("bin_idx").reset_index(drop=True).copy()
        for c in REL_COLS:g[c]=g[c].interpolate(limit_direction="both").fillna(med[c])
        X=xs.transform(g[TRAFFIC_COLS]).astype(np.float32)
        Q=qs.transform(g[REL_COLS]).astype(np.float32)
        yr=g.total.values.astype(float); n=len(g); start=int(n*(CFG["TRAIN_FRAC"]+CFG["VAL_FRAC"]))
        for t in range(max(L,start),n-B*H+1):
            y=np.array([yr[t+h*B:t+(h+1)*B].sum() for h in range(H)],np.float32)
            rows.append({
                "sequence":seq,"t":t,"x":X[t-L:t],"q":Q[t-L:t],"y_raw":y,
                "y":ys.transform(y.reshape(-1,1)).reshape(-1).astype(np.float32)
            })
    return rows

class ExtDS(Dataset):
    def __init__(self,r):self.r=r
    def __len__(self):return len(self.r)
    def __getitem__(self,i):
        a=self.r[i]
        return torch.tensor(a["x"]),torch.tensor(a["q"]),torch.tensor(a["y"]),a["sequence"],int(a["t"])

robustness=pd.DataFrame(); robust_summary=pd.DataFrame()
if CFG["RUN_DEGRADATION"] and len(degraded_df):
    seqs=sorted(degraded_df.sequence.unique())
    scenarios=[("clean",0,clean_df[clean_df.sequence.isin(seqs)])]
    scenarios += [(c,int(s),g) for (c,s),g in degraded_df.groupby(["condition","severity"])]

    rr=[]; mods={
        "Transformer":trained["Transformer"],
        "iTransformer":trained["iTransformer"],
        "Transformer+ReliabilityConcat":trained["Transformer+ReliabilityConcat"],
        "VRA-TFormer":trained["VRA-TFormer"]
    }
    for cond,sev,dfx in tqdm(scenarios,desc="Robustness"):
        rw=ext_windows(dfx)
        if not rw:continue
        loader=DataLoader(ExtDS(rw),batch_size=CFG["BATCH_SIZE"],shuffle=False)
        for name,m in mods.items():
            y,p,_,_=pred(m,loader); t=metrics(y,p,name); t["Condition"]=cond;t["Severity"]=sev;rr.append(t)

    robustness=pd.concat(rr,ignore_index=True)
    robust_summary=robustness.groupby(["Model","Condition","Severity"],as_index=False)["MAE"].mean()
    clean_mae=robust_summary[robust_summary.Condition=="clean"].set_index("Model").MAE.to_dict()
    robust_summary["Relative_Degradation_%"]=robust_summary.apply(
        lambda r:0. if r.Condition=="clean" else
        (r.MAE-clean_mae[r.Model])/max(clean_mae[r.Model],1e-9)*100,axis=1)
    robustness.to_csv(D["tables"]/"table_robustness_full.csv",index=False)
    robust_summary.to_csv(D["tables"]/"table_robustness_summary.csv",index=False)
    display(robust_summary)

In [ ]:
# ========================= WILCOXON + HOLM =========================
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

mean_mae=results.groupby("Model").MAE.mean().sort_values()
cands=[m for m in mean_mae.index if m not in {"VRA-TFormer","Persistence"}]
strongest=cands[0]
print("Strongest comparator:",strongest)

yt,yp=predstore["VRA-TFormer"]; yb,pb=predstore[strongest]
rows=[]; pvals=[]
for j in range(NO):
    ep=np.abs(yt[:,j]-yp[:,j]); eb=np.abs(yb[:,j]-pb[:,j])
    try: stat,p=wilcoxon(eb,ep,alternative="greater")
    except Exception: stat,p=np.nan,1.
    rows.append({
        "Horizon":f"{30*(j+1)}s","Baseline":strongest,
        "Baseline_MAE":eb.mean(),"VRA_MAE":ep.mean(),
        "Improvement_%":(eb.mean()-ep.mean())/max(eb.mean(),1e-9)*100,
        "VRA_win_rate_%":np.mean(ep<eb)*100,
        "Wilcoxon_stat":stat,"p_raw":p
    });pvals.append(p)
reject,ph,_,_=multipletests(pvals,alpha=.05,method="holm")
stats=pd.DataFrame(rows);stats["p_holm"]=ph;stats["significant_0.05"]=reject
stats.to_csv(D["tables"]/"table_wilcoxon_holm.csv",index=False)
display(stats)

In [ ]:
# ========================= LEARNED RELIABILITY =========================
m=trained["VRA-TFormer"];m.eval(); relrows=[]
with torch.no_grad():
    for x,q,y,s,t in ld["test"]:
        pr,r,lam=m(x.to(DEVICE),q.to(DEVICE),return_reliability=True)
        r=r.squeeze(-1).cpu().numpy()
        for i in range(len(s)):
            relrows.append({
                "sequence":s[i],"t":int(t[i]),
                "mean_learned_reliability":float(r[i].mean()),
                "min_learned_reliability":float(r[i].min()),
                "max_learned_reliability":float(r[i].max()),
                "lambda_attention":float(lam.detach().cpu())
            })
learned_rel=pd.DataFrame(relrows)
learned_rel.to_csv(D["tables"]/"learned_reliability_test_windows.csv",index=False)
display(learned_rel.describe())

In [ ]:
# ========================= PAPER FIGURES 4–8 =========================
# Fig 4: horizon MAE
fig,ax=plt.subplots(figsize=(7.2,4))
for name in ["Persistence","LSTM","GRU","Transformer","PatchTST","iTransformer","VRA-TFormer"]:
    z=results[results.Model==name]
    if len(z):ax.plot(z.Horizon,z.MAE,marker="o",lw=1.2,label=name)
ax.set_xlabel("Forecast horizon");ax.set_ylabel("MAE (vehicles / 30-s block)")
ax.set_title("Multi-horizon traffic flow forecasting");ax.legend(ncol=2,frameon=False)
fig.tight_layout();savefig(fig,"Fig4_forecasting_MAE_by_horizon");plt.show()

# Fig 5: ablation
a=ablation.groupby("Model",as_index=False).MAE.mean()
fig,ax=plt.subplots(figsize=(6.3,3.8));ax.bar(a.Model,a.MAE)
ax.set_ylabel("Mean MAE");ax.set_title("Ablation of reliability mechanisms")
ax.tick_params(axis="x",rotation=25);fig.tight_layout();savefig(fig,"Fig5_ablation");plt.show()

# Fig 6: severe degradation
if len(robust_summary):
    z=robust_summary[(robust_summary.Condition!="clean")&(robust_summary.Severity==3)]
    mods=list(z.Model.unique());conds=list(z.Condition.unique());x=np.arange(len(conds));w=.8/max(len(mods),1)
    fig,ax=plt.subplots(figsize=(8,4.2))
    for i,name in enumerate(mods):
        d0=z[z.Model==name].set_index("Condition").reindex(conds)
        ax.bar(x+(i-(len(mods)-1)/2)*w,d0["Relative_Degradation_%"],width=w,label=name)
    ax.set_xticks(x);ax.set_xticklabels(conds,rotation=20)
    ax.set_ylabel("Relative MAE degradation (%)");ax.set_title("Robustness under severe visual degradation")
    ax.legend(frameon=False,ncol=2);fig.tight_layout();savefig(fig,"Fig6_robustness");plt.show()

# Fig 7: 30-s trace
y,p=predstore["VRA-TFormer"];_,bp=predstore[strongest];n=min(120,len(y))
fig,ax=plt.subplots(figsize=(8,3.8))
ax.plot(np.arange(n),y[:n,0],label="Ground truth",lw=1.2)
ax.plot(np.arange(n),bp[:n,0],label=strongest,ls="--",lw=1)
ax.plot(np.arange(n),p[:n,0],label="VRA-TFormer",lw=1.1)
ax.set_xlabel("Test window");ax.set_ylabel("Vehicles / next 30 s");ax.set_title("Representative short-horizon forecast")
ax.legend(frameon=False);fig.tight_layout();savefig(fig,"Fig7_forecast_trace");plt.show()

# Fig 8: convergence
h=histories["VRA-TFormer"]
if len(h):
    fig,ax=plt.subplots(figsize=(5.7,3.6))
    ax.plot(h.epoch,h.train_mse_scaled,label="Train");ax.plot(h.epoch,h.val_mse_scaled,label="Validation")
    ax.set_xlabel("Epoch");ax.set_ylabel("Scaled MSE");ax.set_title("VRA-TFormer convergence")
    ax.legend(frameon=False);fig.tight_layout();savefig(fig,"Fig8_training_convergence");plt.show()

In [ ]:
# ========================= EXCEL + RESULT DIGEST + ZIP =========================
xlsx=D["tables"]/"SmartTraffic_IJIES_All_Tables.xlsx"
with pd.ExcelWriter(xlsx,engine="openpyxl") as w:
    dataset_summary.to_excel(w,sheet_name="Dataset_Summary",index=False)
    split_summary.to_excel(w,sheet_name="Forecast_Splits",index=False)
    results.to_excel(w,sheet_name="Main_Results",index=False)
    efficiency_df.to_excel(w,sheet_name="Efficiency",index=False)
    ablation.to_excel(w,sheet_name="Ablation",index=False)
    stats.to_excel(w,sheet_name="Wilcoxon_Holm",index=False)
    learned_rel.to_excel(w,sheet_name="Learned_Reliability",index=False)
    if len(robustness):
        robustness.to_excel(w,sheet_name="Robustness_Full",index=False)
        robust_summary.to_excel(w,sheet_name="Robustness_Summary",index=False)
    if len(detrac_ap): detrac_ap.to_excel(w,sheet_name="DETRAC_AP",index=False)
    if len(detrac_frames): detrac_frames.to_excel(w,sheet_name="DETRAC_Frame",index=False)

meanres=results.groupby("Model",as_index=False).agg(
    MAE=("MAE","mean"),RMSE=("RMSE","mean"),WAPE_pct=("WAPE_%","mean"),R2=("R2","mean")
).sort_values("MAE")
vra=meanres[meanres.Model=="VRA-TFormer"].iloc[0]
best=meanres.iloc[0]

parts=[
"# SmartTraffic VRA-TFormer — Actual Experiment Digest",
"## Mean forecasting results",meanres.to_markdown(index=False),
"## Wilcoxon-Holm",stats.to_markdown(index=False),
"## Efficiency",efficiency_df.to_markdown(index=False),
"## Ablation",ablation.groupby("Model",as_index=False)[["MAE","RMSE","WAPE_%","R2"]].mean().to_markdown(index=False),
]
if len(robust_summary):
    parts+=["## Robustness",robust_summary.to_markdown(index=False)]
if CFG["RUN_DETRAC"] and (D["tables"]/"table_detrac_perception_summary.csv").exists():
    parts+=["## UA-DETRAC perception validation",
            pd.read_csv(D["tables"]/"table_detrac_perception_summary.csv").to_markdown(index=False)]

parts+=["## Factual manuscript notes",
        f"- Mean VRA-TFormer MAE: **{vra.MAE:.4f}** vehicles / 30-s block.",
        f"- Mean VRA-TFormer WAPE: **{vra.WAPE_pct:.2f}%**.",
        f"- Lowest-MAE model in this run: **{best.Model}** ({best.MAE:.4f}).",
        f"- Strongest non-proposed comparator: **{strongest}**.",
        f"- Learned attention reliability strength λ: **{learned_rel.lambda_attention.iloc[0]:.4f}**."
]
digest=ROOT/"SmartTraffic_IJIES_Result_Digest.md"
digest.write_text("\n\n".join(parts),encoding="utf-8")

manifest={
    "root":str(ROOT),"tables_xlsx":str(xlsx),"digest":str(digest),
    "figures":[str(p) for p in sorted(D["figures"].glob("*"))],
    "tables":[str(p) for p in sorted(D["tables"].glob("*"))],
    "models":[str(p) for p in sorted(D["models"].glob("*"))],
}
(ROOT/"artifact_manifest.json").write_text(json.dumps(manifest,indent=2))

zout=ROOT/"SmartTraffic_IJIES_Paper_Artifacts.zip"
with zipfile.ZipFile(zout,"w",zipfile.ZIP_DEFLATED) as z:
    for key in ["figures","tables","logs"]:
        for p in D[key].rglob("*"):
            if p.is_file():z.write(p,arcname=str(p.relative_to(ROOT)))
    z.write(digest,arcname=digest.name)
    z.write(ROOT/"artifact_manifest.json",arcname="artifact_manifest.json")

print("DONE")
print("Excel:",xlsx)
print("Digest:",digest)
print("ZIP:",zout)